In [1]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.crs import CRS

# ==============================================================================
# USER CONFIGURATION
# ==============================================================================

CONFIG = {
    "master_csv"  : r"E:\Dissertation\Data\GreenbergZhaoRees_river_datasheet.csv",
    "koppen_raster": r"E:\Dissertation\Data\Beck_etal_2018\koppen_geiger_1km.tif",
    "output_csv"  : r"E:\Dissertation\Data\Beck_etal_2018\compiled_koppen_classifications.csv",
}

# Köppen-Geiger integer -> label mapping (Beck et al. 2018)
KOPPEN_LABELS = {
     1: "Af",   2: "Am",   3: "Aw",   4: "BWh",  5: "BWk",
     6: "BSh",  7: "BSk",  8: "Csa",  9: "Csb", 10: "Csc",
    11: "Cwa", 12: "Cwb", 13: "Cwc", 14: "Cfa", 15: "Cfb",
    16: "Cfc", 17: "Dsa", 18: "Dsb", 19: "Dsc", 20: "Dsd",
    21: "Dwa", 22: "Dwb", 23: "Dwc", 24: "Dwd", 25: "Dfa",
    26: "Dfb", 27: "Dfc", 28: "Dfd", 29: "ET",  30: "EF",
}

# ==============================================================================
# COMPILE
# ==============================================================================

river_data = pd.read_csv(CONFIG["master_csv"])
records = []

with rasterio.open(CONFIG["koppen_raster"]) as src:
    raster_crs = src.crs

    for _, row in river_data.iterrows():
        river_name  = row["river_name"]
        working_dir = row["working_directory"]

        shp_path = os.path.join(
            working_dir, "RiverMapping", "Reaches", river_name, f"{river_name}.shp"
        )

        if not os.path.exists(shp_path):
            print(f"Missing shapefile: {river_name}")
            continue

        reaches = gpd.read_file(shp_path)
        if reaches.crs is None:
            reaches = reaches.set_crs("EPSG:4326")

        # Reproject to raster CRS for sampling
        reaches = reaches.to_crs(raster_crs)

        for _, reach_row in reaches.iterrows():
            ds_order = int(reach_row["ds_order"])
            centroid = reach_row.geometry.centroid

            # Sample raster at centroid
            vals = list(src.sample([(centroid.x, centroid.y)]))
            koppen_int = int(vals[0][0])
            koppen_class = KOPPEN_LABELS.get(koppen_int, "unknown")

            records.append({
                "river_name"  : river_name,
                "ds_order"    : ds_order,
                "koppen_int"  : koppen_int,
                "koppen_class": koppen_class,
            })

compiled = pd.concat([pd.DataFrame([r]) for r in records], ignore_index=True)
compiled.to_csv(CONFIG["output_csv"], index=False)

print(f"Compiled {len(compiled)} reaches across {compiled['river_name'].nunique()} rivers.")
print(f"\nKöppen distribution:")
print(compiled["koppen_class"].value_counts().to_string())
print(f"\nSaved to: {CONFIG['output_csv']}")
compiled.head(10)


Compiled 153 reaches across 125 rivers.

Köppen distribution:
koppen_class
Aw     25
BSh    20
Cwa    19
Af     19
Am     16
Cfa    15
Dfc    12
Dfb     9
BWh     3
BSk     3
Dsc     2
Dfd     2
Dwb     2
BWk     1
Dfa     1
Cfb     1
Csa     1
Dsb     1
Dwa     1

Saved to: E:\Dissertation\Data\Beck_etal_2018\compiled_koppen_classifications.csv


,river_name,ds_order,koppen_int,koppen_class
0,Aladan_VerkhoyanskiyPerevoz,1,28,Dfd
1,Amazonas_Jatuarana,1,1,Af
2,Amazonas_Tamshiyacu,1,1,Af
3,AmuDarya_Kerki,1,4,BWh
4,Amur_Khabarovsk,1,22,Dwb
5,Amur_Komsomolsk,1,22,Dwb
6,Amyl_Kachulka,1,26,Dfb
7,Apalachicola_NearBlountstown,1,14,Cfa
8,Araguaia_Aruana,1,3,Aw
9,Araguaia_LuizAlves,1,3,Aw
